### Manual: 14-forward

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Forward one sample: shapes, finiteness of every stage, depth-channel check.

Runs `step_forward.py` in the pinned venv. It hooks `dtransform` to capture the
exact depth tensor the transform receives, hooks the camera BEV, fused BEV,
decoder neck and head outputs, runs the model in eval mode under `no_grad`, and
records a pass/fail verdict per acceptance criterion.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("step_forward.py", log_name="step_forward.log", timeout=3600)
report = sc.load_json("forward.json")
print()
print("rc = %d" % rc)
print("batch shapes : %s" % json.dumps(report.get("batch", {}).get("shapes")))
print("forward      : %s" % json.dumps(report.get("forward")))
depth = report.get("depth_check", {})
print()
print("DEPTH CHECK")
for key in (
    "dtransform_first_layer",
    "dtransform_accepted_channels",
    "depth_tensor_shape",
    "depth_tensor_channels",
    "depth_tensor_nonzero_fraction",
    "depth_tensor_finite",
    "depth_tensor_min",
    "depth_tensor_max",
    "image_size",
    "depth_spatial_size",
    "depth_spatial_matches_image_size",
    "dtransform_output_size_computed",
    "feature_size",
    "dtransform_output_matches_feature_size",
    "channels_match",
    "V_D",
    "V_C",
    "class_source_lines",
    "forward_source_lines",
    "pass",
):
    print("  %-40s %s" % (key, json.dumps(depth.get(key))))
finite = report.get("finite_check", {})
print()
print("FINITE CHECK")
print("  stages : %s" % json.dumps(finite.get("stages")))
print("  head   : %s" % json.dumps(finite.get("head")))
print("  outputs: %s" % json.dumps(finite.get("outputs")))
print("  pass   : %s" % json.dumps(finite.get("pass")))
print()
print("OUTPUTS: %s" % json.dumps(report.get("outputs"), sort_keys=True))
print("VERDICT: %s" % json.dumps(report.get("pass"), sort_keys=True))
print("FORWARD_CELL_OK" if report.get("all_pass") else "FORWARD_CELL_FAILED (see logs/step_forward.log)")
